# LPLH Framework — Google Colab Runner

Runs the **Learning to Play Like Humans** (ACL 2025) framework on Colab GPUs using **qwen2.5:7b** (optimised for T4 free tier).

## Before you start
1. **Set runtime to GPU**: Runtime → Change runtime type → **T4 GPU** (free) or A100 (Pro)
2. **Have `zork1.z5` ready** — you will upload it in Step 6
3. Run cells **in order**, top to bottom

## Expected runtimes on T4 GPU (qwen2.5:7b)
| Config | Total steps | Approx. time |
|---|---|---|
| 3 epochs × 100 steps | 300 | ~45 min |
| 3 epochs × 250 steps | 750 | ~2 h |
| 10 epochs × 250 steps | 2500 | ~6 h |

> **Note**: Free Colab sessions disconnect after ~12 h of GPU use. 10 epochs fits comfortably within this limit with qwen2.5:7b.

## Paper baseline (Zork1)
- Qwen2.5-7B + LPLH: avg **9**, max **15**
- Qwen2.5-14B + LPLH: avg **14.3**, max **20** *(use A100/Pro+ for 14b)*

---
## Step 1 — Check GPU

In [ ]:
import subprocess, sys

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('ERROR: No GPU detected.')
    print('Go to Runtime > Change runtime type and select T4 GPU, then reconnect.')
    sys.exit(1)

gpu_name, vram_str = result.stdout.strip().split(',')
vram_mb = int(vram_str.strip().split()[0])
print(f'GPU : {gpu_name.strip()}')
print(f'VRAM: {vram_mb} MB ({vram_mb/1024:.1f} GB)')

if vram_mb < 10000:
    print('WARNING: Less than 10 GB VRAM. qwen2.5:14b may OOM.')
    print('Consider switching to qwen2.5:7b in Step 6.')
else:
    print('VRAM OK — qwen2.5:14b will fit.')

---
## Step 2 — Install system dependencies

In [ ]:
%%bash
set -e
apt-get update -qq
# pciutils provides lspci — required for Ollama to auto-detect the GPU
apt-get install -y -qq build-essential python3-dev libncurses5-dev libncursesw5-dev curl zstd pciutils
echo 'System dependencies installed.'

---
## Step 3 — Install Python packages

> You may see OpenTelemetry version conflict warnings at the end — these are pre-existing Colab conflicts and can be safely ignored.

In [ ]:
%%bash
pip install -q jericho ollama chromadb sentence-transformers
echo 'Python packages installed.'

---
## Step 4 — Install Ollama

The install script may print a `systemd` warning — this is expected on Colab and harmless. The binary installs correctly regardless.

In [ ]:
%%bash
# zstd: required by Ollama installer to extract binary
# pciutils: provides lspci so Ollama can detect and use the T4 GPU
apt-get install -y -qq zstd pciutils

# The install script exits non-zero when it can't register a systemd service.
# That's expected on Colab — the binary installs fine. We verify with `ollama --version`.
curl -fsSL https://ollama.com/install.sh | sh || true

if command -v ollama &> /dev/null; then
    echo "Ollama installed: $(ollama --version 2>&1 | tail -1)"
else
    echo 'ERROR: ollama binary not found after install. Check output above.'
    exit 1
fi

---
## Step 5 — Start Ollama server and pull qwen2.5:14b

This is the paper's main model (Qwen2.5-14B-Instruct). Download is ~8.5 GB — allow ~5 min.

In [ ]:
import subprocess, time, urllib.request, sys

# Start Ollama server in background
print('Starting Ollama server...')
get_ipython().system_raw('ollama serve > /tmp/ollama.log 2>&1 &')

# Poll until the server responds
for i in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2)
        print(f'Server ready (after {i+1}s).')
        break
    except Exception:
        time.sleep(1)
else:
    print('ERROR: Ollama server did not start in 30s. Server log:')
    subprocess.run(['cat', '/tmp/ollama.log'])
    sys.exit(1)

# Pull the model (streaming output so progress is visible)
MODEL = 'qwen2.5:7b'
print(f'Pulling {MODEL}... (~4.5 GB, may take ~3 min)')
result = subprocess.run(['ollama', 'pull', MODEL])
if result.returncode != 0:
    print(f'ERROR: Failed to pull {MODEL}.')
    sys.exit(1)
print(f'{MODEL} ready.')

In [ ]:
# Smoke test — verify the model responds correctly
import ollama

MODEL = 'qwen2.5:7b'
print(f'Testing {MODEL}...')
try:
    resp = ollama.chat(
        model=MODEL,
        messages=[{'role': 'user', 'content': 'Reply with exactly: OK'}],
        options={'temperature': 0.0}
    )
    content = resp['message']['content'].strip()
    print(f'Model response: "{content[:80]}"')
    print('Smoke test passed.')
except Exception as e:
    print(f'ERROR: {e}')

---
## Step 6 — Clone repository & upload game file

In [ ]:
%%bash
set -e
REPO='https://github.com/ogulcaanozen/LPLH-InteractiveFiction.git'
DEST='LPLH-InteractiveFiction'

if [ -d "$DEST" ]; then
    echo 'Repo already exists. Pulling latest changes...'
    git -C "$DEST" pull origin master
else
    echo 'Cloning repository...'
    git clone "$REPO" "$DEST"
fi

mkdir -p "$DEST/games" "$DEST/data/logs"
echo 'Repository ready.'

In [ ]:
# Upload zork1.z5
# Option A (this cell): use the file picker.
# Option B: drag-and-drop zork1.z5 into LPLH-InteractiveFiction/games/ in the left sidebar,
#            then re-run this cell to verify.

import os
from google.colab import files

GAME_PATH = 'LPLH-InteractiveFiction/games/zork1.z5'

if os.path.exists(GAME_PATH):
    size_kb = os.path.getsize(GAME_PATH) / 1024
    print(f'zork1.z5 already present ({size_kb:.0f} KB). Skipping upload.')
else:
    print('Select zork1.z5 from your computer...')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        with open(GAME_PATH, 'wb') as f:
            f.write(data)
        print(f'Saved: {GAME_PATH} ({len(data)/1024:.0f} KB)')

assert os.path.exists(GAME_PATH), 'ERROR: zork1.z5 not found. Upload it before continuing.'
print('Game file OK.')

---
## Step 7 — Configure experiment

Edit the values below, then run the cell.

In [ ]:
import os

# ── Experiment settings ────────────────────────────────────────────────────
NUM_EPOCHS = 10   # Paper uses 10. Reduce to 3 for a quick validation run.
MAX_STEPS  = 250  # Paper uses 250 steps per epoch.
GAME       = 'zork1'

# ── Model settings ─────────────────────────────────────────────────────────
# 'qwen2.5:7b'   — recommended for T4 (4.5 GB, safe on 16 GB VRAM)
# 'qwen2.5:14b'  — paper's exact model (8.5 GB, use on A100/Pro+ only)
LLM_MODEL = 'qwen2.5:7b'

# Apply via env vars — overrides config.py defaults without editing any file
os.environ['LPLH_LLM_PROVIDER'] = 'ollama'
os.environ['LPLH_LLM_MODEL']    = LLM_MODEL

print(f'Epochs : {NUM_EPOCHS}')
print(f'Steps  : {MAX_STEPS} per epoch ({NUM_EPOCHS * MAX_STEPS} total)')
print(f'Game   : {GAME}')
print(f'Model  : {LLM_MODEL}')

---
## Step 8 — Run the experiment

Output streams in real time. A structured run log is also written to `data/logs/run_log_*.txt`.

You can safely interrupt with the **Stop** button — results up to that point are saved.

In [ ]:
import subprocess, sys, os

GAME_PATH = 'LPLH-InteractiveFiction/games/zork1.z5'
if not os.path.exists(GAME_PATH):
    print('ERROR: zork1.z5 not found. Run Step 6 first.')
    sys.exit(1)

cmd = [
    sys.executable, 'run_game.py',
    '--game',   GAME,
    '--epochs', str(NUM_EPOCHS),
    '--steps',  str(MAX_STEPS),
    '--verbose',
]

print('Command:', ' '.join(cmd))
print('=' * 60)

proc = subprocess.Popen(
    cmd,
    cwd='LPLH-InteractiveFiction',
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    print('\nRun interrupted by user. Partial results are saved in data/.')

print('=' * 60)
print(f'Exit code: {proc.returncode}')

---
## Step 9 — Download results

In [ ]:
import subprocess, os, glob
from google.colab import files

# Show what we have
all_files = sorted(glob.glob('LPLH-InteractiveFiction/data/**/*', recursive=True))
data_files = [f for f in all_files if os.path.isfile(f)]

print(f'Result files ({len(data_files)} total):')
for f in data_files:
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f} ({size_kb:.1f} KB)')

# Zip and download
zip_name = 'lplh_results.zip'
subprocess.run(['zip', '-r', zip_name, 'LPLH-InteractiveFiction/data/'], check=True)
print(f'\nDownloading {zip_name}...')
files.download(zip_name)